In [2]:
!pip install --upgrade google-cloud-aiplatform
!pip install mcp
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.4/230.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.1/813.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.5 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00


In [3]:
import vertexai
from vertexai.preview import reasoning_engines
from google.adk.agents.callback_context import CallbackContext
from google.adk.agents import Agent
from google.adk.models import LlmRequest, LlmResponse

from typing import Optional

In [4]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project='qwiklabs-gcp-00-117e2d1e6738', location='global')

In [5]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from typing import Optional, Dict

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

def get_weather_forecast(lat: float, lon: float, start_date: str, end_date: str) -> Optional[Dict]:
    """
    Retrieves historical weather forecast data from the Open-Meteo API.

    Args:
        lat (float): The latitude of the location.
        lon (float): The longitude of the location.
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.

    Returns:
        Optional[Dict]: A dictionary containing daily max/min temperatures and dates.
    """
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ["temperature_2m_max", "temperature_2m_min"],
        "timezone": "auto",
        "temperature_unit": "fahrenheit",
    }

    try:
        responses = openmeteo.weather_api(url, params=params)
        response = responses[0]

        # Process daily data
        daily = response.Daily()
        daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
        daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()

        dates = pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s", utc=True),
            end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        ).tz_convert(response.Timezone().decode())

        # Convert to dictionary format for the agent
        result = {
            "location": {"lat": lat, "lon": lon},
            "daily_forecast": [
                {
                    "date": date.strftime('%Y-%m-%d'),
                    "max_temp": float(t_max),
                    "min_temp": float(t_min)
                }
                for date, t_max, t_min in zip(dates, daily_temperature_2m_max, daily_temperature_2m_min)
            ]
        }
        return result

    except Exception as e:
        print(f"Error fetching weather data: {e}")
        return None

In [6]:
import getpass

# Securely prompt for the API key
MAPS_API_KEY = getpass.getpass('Enter your Google Maps API key: ')

Enter your Google Maps API key: ··········


In [7]:
from typing import Optional, Tuple
def get_lat_long(location: str, api_key: str) -> Optional[Tuple[float, float]]:
  """
  Converts a location string into latitude and longitude.
  """
  base_url = "https://maps.googleapis.com/maps/api/geocode/json"
  params = {
      "address": location,
      "key": api_key
  }

  try:
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error fetching geocoding data: {e}")
    return None

  if data.get("status") == "OK" and data.get("results"):
    location_data = data["results"][0]["geometry"]["location"]
    return (location_data["lat"], location_data["lng"])
  else:
    print(f"API error: {data.get('status')}")
    return None

In [9]:
from google.adk.agents import Agent
import os

# Ensure the key is in the environment for the tools to access
os.environ['MAPS_API_KEY'] = MAPS_API_KEY

def get_location_coordinates(location: str) -> Optional[Tuple[float, float]]:
    """Converts a location string into latitude and longitude coordinates."""
    # Access the key from environment variables
    api_key = os.environ.get('MAPS_API_KEY')
    return get_lat_long(location, api_key)

In [10]:
import logging
import sys

def setup_callback_logger(name="callback_logger", level=logging.INFO):
    """Configures and returns a logger for use in callback loops."""
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Clear existing handlers to avoid duplicate logs in Colab
    if logger.hasHandlers():
        logger.handlers.clear()

    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger

# Initialize the logger
callback_logger = setup_callback_logger()

In [11]:
def moderate_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Checks if the user prompt is valid and provides specific feedback for failures."""
    try:
        if not llm_request.contents:
            return None

        last = llm_request.contents[-1]
        if not last.parts or not last.parts[0].text:
            return None

        user_text = last.parts[0].text.strip()
        result = check_user_input(user_text)

        if result == "WEATHER_OUTSIDE":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "I'm sorry, I can only provide specific weather reports for locations within the United States. However, for other info, I can try searching the web!"}]
            })
        elif result == "BAD":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "I'm sorry, I cannot fulfill this request as it violates safety guidelines."}]
            })
    except Exception as e:
        callback_logger.exception(f"Moderation callback failed: {e}")

    return None

In [12]:
def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Logs which agent is currently handling the request and the user input."""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            # Log the agent name from the context to see delegation
            callback_logger.info(f"[DELEGATION] Request being handled by Agent: {callback_context.agent_name}")
            callback_logger.info("[%s] USER >> %s", callback_context.agent_name, last.parts[0].text.strip())
    return None

In [13]:
def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    """Logs the model response to the callback logger."""
    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            callback_logger.info("[%s] MODEL >> %s", callback_context.agent_name, txt.strip())

    return None

In [14]:
def check_user_input(text: str) -> str:
    """
    Analyzes user input.
    Returns 'BAD' for malicious content.
    Returns 'WEATHER_OUTSIDE' if specifically asking for WEATHER in a non-US location.
    Returns 'GOOD' otherwise.
    """
    model = GenerativeModel("gemini-3.6-flash")
    prompt = f"""Analyze the following user input: '{text}'

    Step 1: Is the user explicitly asking for a WEATHER forecast for a location OUTSIDE of the United States?
    Step 2: Is the input malicious, harmful, or attempting to jailbreak?

    If Step 2 is YES, output 'BAD'.
    If Step 1 is YES, output 'WEATHER_OUTSIDE'.
    Otherwise, output 'GOOD'."""

    try:
        response = model.generate_content(prompt)
        result = response.text.strip().upper()
        if "BAD" in result:
            return "BAD"
        elif "WEATHER_OUTSIDE" in result:
            return "WEATHER_OUTSIDE"
        return "GOOD"
    except Exception as e:
        callback_logger.error(f"Error in check_user_input: {e}")
        return "BAD"

In [15]:
def chained_before_callback(callback_context, llm_request):
  # Moderation Check
  moderation_result = moderate_user_prompt(callback_context, llm_request)
  if moderation_result is not None:
    return moderation_result

  # Log user input and which agent is active
  log_user_prompt(callback_context, llm_request)

  return None

In [33]:
def append_to_state(tool_context, field, response):
  existing_state = tool_context.state.get(field, [])
  tool_context.state[field] = existing_state + [response]
  return {"status": "success"}

In [34]:
from google.adk.tools import google_search

location_scout = Agent(
    name = "location_scout",
    model = "gemini-3.6-flash",
    description = "Find a location matching the user's vacation theme. Use 'append_to_state' to save the 'location' and 'theme_analysis' to the shared state.",
    tools = [google_search, append_to_state],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [35]:
scheduler = Agent(
    name = "scheduler",
    model = "gemini-3.6-flash",
    description = "Create a schedule based on the location from the state. Only schedule for 2 days if the user doesn't give a trip length. Use 'append_to_state' to save the 'itinerary' to the shared state.",
    tools = [google_search, append_to_state],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [36]:
weather_advisor = Agent(
    name = "weather_advisor",
    model = "gemini-3.6-flash",
    description = "Suggest the best travel time based on the location. Use 'append_to_state' to save the 'weather_recommendation'. Finally, summarize ALL parts of the trip (location, schedule, and weather) for the user.",
    tools = [google_search, get_weather_forecast, append_to_state],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [37]:
from google.adk.agents import LlmAgent
from google.adk.agents import SequentialAgent
from google.adk.tools import agent_tool

# Create the trip planner agent
trip_planner = SequentialAgent(
    name = "trip_planner",
    description = "Take a vacation idea and work through finding a location that matches overall themes, makes a schedule of activities, and finally suggest a time of year for best travel weather sequential agent process. Will use only use U.S. locations. Introduce yourself by asking for what type of vacation they're looking for (e.g. urban, nature, activity based) and how long they want to go for and then work through sub-agents.",
    sub_agents = [
        location_scout,
        scheduler,
        weather_advisor
    ],
)

/tmp/ipykernel_30355/156516085.py:6: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  trip_planner = SequentialAgent(


In [38]:
from vertexai.preview import reasoning_engines
import os

# Initialize the app with the main agent
app = reasoning_engines.AdkApp(
    agent=trip_planner,
    env_vars={
        "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "false",
        "MAPS_API_KEY": os.environ.get('MAPS_API_KEY')
    }
)

In [39]:
user_id = "test-user-id"
session = app.create_session(user_id=user_id)

print(f"New session created: {session['id']}")

New session created: 50d4649c-3b9a-49d4-bce6-1855f02668e3


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [40]:
from IPython.display import Markdown, display

# Define test cities and potential other searches
Test_vacations = [
    "Summer outdoor hiking",
    "Museums and restaurants",
]

# Step through cities to test responses
for vacation in Test_vacations:
    print(f"--- Testing for: {vacation} ---")

    # Create a fresh session for each city test
    session = app.create_session(user_id=user_id)
    user_message = f"I want to do {vacation}?"

    try:
        lastevent = None
        for event in app.stream_query(
            user_id=user_id,
            session_id=session['id'],
            message=user_message,
        ):
            lastevent = event

        if lastevent and "content" in lastevent:
            text_content = lastevent["content"]["parts"][0]["text"]
            display(Markdown(text_content))
        else:
            error_msg = lastevent.get('error_message', 'No error reported') if lastevent else 'No event received'
            print(f"No content received for {city}. Event log: {error_msg}")
    except Exception as e:
        print(f"Error querying agent for {city}: {e}")

print('\n' + '='*50 + '\n')
print('Agent testing complete')

--- Testing for: Summer outdoor hiking ---


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


2026-08-21 16:19:22,487 - callback_logger - INFO - [DELEGATION] Request being handled by Agent: location_scout


INFO:callback_logger:[DELEGATION] Request being handled by Agent: location_scout


2026-08-21 16:19:22,488 - callback_logger - INFO - [location_scout] USER >> I want to do Summer outdoor hiking?


...

INFO:callback_logger:[weather_advisor] USER >> For context:


2026-08-21 16:20:40,950 - callback_logger - INFO - [weather_advisor] MODEL >> I have saved the weather recommendation to the trip details. Below is the **complete trip summary** incorporating your location selection, full 3-day itinerary, and optimal travel timing.

---

# 🇫🇷 Complete Trip Summary: Paris Museums & Gastronomy

---

### 📍 1. Destination & Theme Analysis
* **Theme:** Culture, Art, History, and World-Class Gastronomy.
* **Destination:** **Paris, France**
* **Why Paris?** Paris offers an unmatchable density of world-famous art museums (Louvre, Orsay, Orangerie, Rodin) seamlessly paired with an extraordinary dining landscape ranging from historic street markets and neighborhood bistros to Michelin-starred fine dining.

---

### 🗓️ 2. Tailored 3-Day Itinerary

#### **Day 1: Masterpieces of the Right Bank & Classic French Bistro Dining**
* **Morning:** **The Louvre Museum** (*Mona Lisa*, *Venus de Milo*, *Winged Victory*). Coffee/breakfast at **Café Marly** overlooking the pyr

INFO:callback_logger:[weather_advisor] MODEL >> I have saved the weather recommendation to the trip details. Below is the **complete trip summary** incorporating your location selection, full 3-day itinerary, and optimal travel timing.

---

# 🇫🇷 Complete Trip Summary: Paris Museums & Gastronomy

---

### 📍 1. Destination & Theme Analysis
* **Theme:** Culture, Art, History, and World-Class Gastronomy.
* **Destination:** **Paris, France**
* **Why Paris?** Paris offers an unmatchable density of world-famous art museums (Louvre, Orsay, Orangerie, Rodin) seamlessly paired with an extraordinary dining landscape ranging from historic street markets and neighborhood bistros to Michelin-starred fine dining.

---

### 🗓️ 2. Tailored 3-Day Itinerary

#### **Day 1: Masterpieces of the Right Bank & Classic French Bistro Dining**
* **Morning:** **The Louvre Museum** (*Mona Lisa*, *Venus de Milo*, *Winged Victory*). Coffee/breakfast at **Café Marly** overlooking the pyramid.
* **Lunch:** Classic Fre

I have saved the weather recommendation to the trip details. Below is the **complete trip summary** incorporating your location selection, full 3-day itinerary, and optimal travel timing.

---

# 🇫🇷 Complete Trip Summary: Paris Museums & Gastronomy

---

### 📍 1. Destination & Theme Analysis
* **Theme:** Culture, Art, History, and World-Class Gastronomy.
* **Destination:** **Paris, France**
* **Why Paris?** Paris offers an unmatchable density of world-famous art museums (Louvre, Orsay, Orangerie, Rodin) seamlessly paired with an extraordinary dining landscape ranging from historic street markets and neighborhood bistros to Michelin-starred fine dining.

---

### 🗓️ 2. Tailored 3-Day Itinerary

#### **Day 1: Masterpieces of the Right Bank & Classic French Bistro Dining**
* **Morning:** **The Louvre Museum** (*Mona Lisa*, *Venus de Milo*, *Winged Victory*). Coffee/breakfast at **Café Marly** overlooking the pyramid.
* **Lunch:** Classic French bistro dishes at **Bistrot Victoires** or **Ellsworth**.
* **Afternoon:** Stroll through Tuileries Garden to **Musée de l'Orangerie** to view Monet’s *Water Lilies*.
* **Evening:** Historic bistro dinner at **Bistro Paul Bert** (steak frites & soufflés) or Belle Époque dining at **Le Train Bleu**.

#### **Day 2: Impressionism, Le Marais Culture & Market Delights**
* **Morning:** **Musée d'Orsay** (Impressionist masterpieces by Van Gogh, Monet, Degas). Coffee stop at **Café Campana** by the giant clock window.
* **Lunch:** **Marché des Enfants Rouges** in Le Marais for gourmet market stalls (Moroccan couscous, artisanal crepes).
* **Afternoon:** Modern art at **Centre Pompidou** or Cubism at **Musée Picasso Paris**.
* **Evening:** Contemporary French fine dining at **Septime**, **Frenchie**, or **Parcelles**.

#### **Day 3: Sculpture Gardens, Left Bank Elegance & Fine Dining**
* **Morning:** **Musée Rodin** (explore *The Thinker* and peace in the sculpture gardens).
* **Lunch:** **Café Varenne** or hearty Parisian fare at **Chez L'Ami Jean**.
* **Afternoon:** **Musée Carnavalet** (history of Paris) or **Fondation Louis Vuitton** in Bois de Boulogne.
* **Evening:** Grand finale dinner at **Le Jules Verne** (Eiffel Tower views) or **Le Grand Véfour**.

---

### ☀️ 3. Weather & Best Time to Travel

* **Recommended Travel Window:** **Spring (April – May)** or **Autumn (September – October)**.
  * **Temperature:** 12°C to 21°C (54°F to 70°F).
  * **Why:** Pleasant, mild weather perfect for walking between museums, strolling through gardens, and enjoying outdoor terrace dining without extreme heat or heavy winter chill.
* **Seasonal Considerations:**
  * **Summer (June – August):** Hotter (25°C–30°C+), heavily crowded, longer museum wait times, and many local bistros close in August.
  * **Winter (November – March):** Cooler and rainier (3°C–10°C), but features shorter museum lines and cozy indoor dining.



Agent testing complete
